In [ ]:
# ============================================================
# PLOTS:
# 1. Global polar order parameter, Phi
# 2. Polar order parameter inside the non-noisy region, Phi_in
# 3. Normalized particle density inside the non-noisy region, rho_in/rho_0
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D


# ============================================================
# JOURNAL STYLE
# ============================================================

plt.rcParams.update({'font.family': 'C059', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.linewidth': 1.4, 'mathtext.fontset': 'stix', 'xtick.direction': 'in', 'ytick.direction': 'in', 'xtick.top': True, 'ytick.right': True})


# ============================================================
# PLOT SETTINGS
# ============================================================

FIGSIZE = (3.5, 2.8)
LABEL_FONTSIZE = 14
TICK_FONTSIZE = 10
LEGEND_FONTSIZE = 10
LINE_WIDTH = 1.2
MARKER_SIZE = 7
MARKER_EDGEW = 0.5


# ============================================================
# USER CONFIGURATION
# ============================================================

INPUT_CSV = "input_results.csv"
OUTPUT_PDF = "analysis_plots.pdf"
ALLOWED_SRADS = np.array([0, 2, 4, 6, 8])
RHO_TOTAL = 2.5
L = 20.0


# ============================================================
# LOAD DATA
# ============================================================

data = np.loadtxt(INPUT_CSV, delimiter=",", skiprows=1)
if data.ndim == 1:
    data = data.reshape(1, -1)

fac = data[:, 0]
srad = data[:, 1]
phi = data[:, 2]
phi_in = data[:, 4]
N_in = data[:, 8]


# ============================================================
# CALCULATE INSIDE DENSITY
# ============================================================

inside_area = np.pi * srad**2
rho_in = np.where(srad > 0, N_in / inside_area, 0.0)
rho_in_ratio = rho_in / RHO_TOTAL


# ============================================================
# STYLE MAP
# ============================================================

srad_style = {0: {'color': 'black', 'marker': '+', 'label': 's = 0'}, 2: {'color': 'red', 'marker': 'x', 'label': 's = 0.1'}, 4: {'color': 'blue', 'marker': '*', 'label': 's = 0.2'}, 6: {'color': 'green', 'marker': 'o', 'label': 's = 0.3'}, 8: {'color': 'purple', 'marker': '^', 'label': 's = 0.4'}}


# ============================================================
# LEGEND
# ============================================================

def add_legend(ax):
    handles = [Line2D([0], [0], color=srad_style[s]['color'], marker=srad_style[s]['marker'], linewidth=LINE_WIDTH, markersize=MARKER_SIZE, markeredgewidth=MARKER_EDGEW, label=srad_style[s]['label']) for s in ALLOWED_SRADS]
    leg = ax.legend(handles=handles, ncol=1, loc="center right", bbox_to_anchor=(0.98, 0.5), fontsize=LEGEND_FONTSIZE, frameon=True, handlelength=1.2, handletextpad=0.35, labelspacing=0.25, borderpad=0.3)
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_linewidth(1.0)


# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_panel(x, y, ylabel, ylim, allowed_srads):
    fig, ax = plt.subplots(figsize=FIGSIZE)

    for s in allowed_srads:
        mask = srad == s
        if np.any(mask):
            idx = np.argsort(x[mask])
            ax.plot(x[mask][idx], y[mask][idx], color=srad_style[s]['color'], marker=srad_style[s]['marker'], markersize=MARKER_SIZE, markeredgewidth=MARKER_EDGEW, linewidth=LINE_WIDTH)

    ax.set_xlabel(r'$\eta$', fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_ylim(*ylim)
    ax.tick_params(axis="both", which="major", direction="in", length=5, width=1.4, labelsize=TICK_FONTSIZE, top=True, right=True)
    ax.grid(axis="y", linewidth=0.6, alpha=0.3)
    ax.grid(axis="x", visible=False)
    add_legend(ax)
    plt.tight_layout()

    return fig


# ============================================================
# GENERATE MULTI-PAGE PDF
# ============================================================

with PdfPages(OUTPUT_PDF) as pdf:
    pdf.savefig(plot_panel(fac, phi, r'$\Phi$', (0, 1.1), ALLOWED_SRADS))
    plt.close()

    pdf.savefig(plot_panel(fac, phi_in, r'$\Phi_{\mathrm{in}}$', (0, 1.1), ALLOWED_SRADS[1:]))
    plt.close()

    pdf.savefig(plot_panel(fac, rho_in_ratio, r'$\frac{\rho_{\mathrm{in}}}{\rho_0}$', (0, 1.5), ALLOWED_SRADS[1:]))
    plt.close()

print(OUTPUT_PDF)